# End-to-End Commodity Allocation System

**Goal:** Build a complete, production-ready commodity allocation system that runs weekly with full logging, monitoring, and guardrails.

**What you'll build:**
- Complete pipeline: data loading → feature engineering → contextual bandit → allocation
- Production guardrails: position limits, stop-loss, circuit breakers
- Weekly reporting: allocation, performance, confidence levels
- Deployment instructions for automated weekly runs

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from collections import defaultdict, deque
import json

np.random.seed(42)

In [ ]:
learning_objectives(["**Foundations** (Module 0)", "**Core Algorithms** (Modules 1-3)", "**Applications** (Modules 4-5)", "**Advanced Topics** (Module 6)", "**Production Systems** (Module 7)", "Deploy the commodity allocator with paper trading (no real capital)"])

In [ ]:
apply_course_theme()
apply_plot_theme()

## Step 1: Complete Production System Architecture

In [ ]:
class CommodityAllocationSystem:
    """Production-ready commodity allocation with bandits."""

    def __init__(self, commodities, initial_capital=100000):
        self.commodities = commodities
        self.capital = initial_capital

        # Contextual bandit (LinUCB-style with Thompson Sampling)
        self.feature_dim = 5  # VIX, momentum, volatility, term_structure, regime
        self.alpha = {c: np.ones(self.feature_dim) for c in commodities}
        self.beta = {c: np.ones(self.feature_dim) for c in commodities}

        # Guardrails
        self.max_position_pct = 0.40  # Max 40% in any single commodity
        self.stop_loss_pct = -0.15     # Stop if down 15% from peak
        self.disabled_commodities = set()

        # Logging and monitoring
        self.decision_log = []
        self.performance_log = []
        self.alerts = []

        # State tracking
        self.peak_capital = initial_capital
        self.current_allocation = None

    def get_market_features(self, prices_df, week):
        """Extract contextual features from market data."""
        features = {}

        for commodity in self.commodities:
            if commodity not in prices_df.columns:
                continue

            # Get recent price history
            recent_prices = prices_df[commodity].iloc[max(0, week-20):week+1]

            if len(recent_prices) < 5:
                # Insufficient data, use neutral features
                features[commodity] = np.array([0.5, 0.5, 0.5, 0.5, 0.5])
                continue

            # Feature 1: Short-term momentum (4-week return)
            momentum = (recent_prices.iloc[-1] / recent_prices.iloc[-5] - 1) if len(recent_prices) >= 5 else 0
            momentum_feature = 1 / (1 + np.exp(-momentum * 10))  # Sigmoid

            # Feature 2: Volatility (normalized)
            volatility = recent_prices.pct_change().std()
            vol_feature = min(volatility / 0.1, 1.0)  # Cap at 10% weekly vol

            # Feature 3: Mean reversion (distance from MA)
            ma = recent_prices.mean()
            mean_rev = (recent_prices.iloc[-1] - ma) / (ma + 1e-8)
            mean_rev_feature = 1 / (1 + np.exp(-mean_rev * 5))

            # Feature 4: Trend strength (correlation with time)
            if len(recent_prices) >= 10:
                trend = np.corrcoef(np.arange(len(recent_prices)), recent_prices)[0, 1]
                trend_feature = (trend + 1) / 2  # Map [-1,1] to [0,1]
            else:
                trend_feature = 0.5

            # Feature 5: Regime indicator (high/low volatility)
            regime = 1.0 if volatility > 0.05 else 0.0

            features[commodity] = np.array([
                momentum_feature,
                vol_feature,
                mean_rev_feature,
                trend_feature,
                regime
            ])

        return features

    def select_commodity(self, features):
        """Select commodity using contextual Thompson Sampling."""
        available = [c for c in self.commodities if c not in self.disabled_commodities]

        if not available:
            return "CASH"  # All commodities disabled

        # Thompson Sampling: sample expected returns for each commodity
        scores = {}
        for commodity in available:
            if commodity not in features:
                scores[commodity] = 0.0
                continue

            # Sample from Beta distribution for each feature
            feature_vec = features[commodity]
            sampled_weights = np.array([
                np.random.beta(self.alpha[commodity][i], self.beta[commodity][i])
                for i in range(self.feature_dim)
            ])

            # Dot product for linear score
            scores[commodity] = np.dot(feature_vec, sampled_weights)

        # Select best
        selected = max(scores, key=scores.get)
        return selected, scores

    def apply_guardrails(self, selected, features, week):
        """Validate and potentially override commodity selection."""
        override = False
        reason = None

        # Guardrail 1: Disabled commodities
        if selected in self.disabled_commodities:
            selected = "CASH"
            override = True
            reason = "Commodity disabled"

        # Guardrail 2: High volatility override
        if selected in features and features[selected][1] > 0.8:  # Vol feature > 0.8
            selected = "CASH"
            override = True
            reason = "Excessive volatility"

        # Guardrail 3: Stop-loss check
        drawdown = (self.capital - self.peak_capital) / self.peak_capital
        if drawdown < self.stop_loss_pct:
            selected = "CASH"
            override = True
            reason = f"Stop-loss triggered ({drawdown:.1%} drawdown)"
            self.alerts.append(f"Week {week}: STOP-LOSS TRIGGERED")

        return selected, override, reason

    def update(self, commodity, features, reward):
        """Update bandit model with observed reward."""
        if commodity == "CASH" or commodity not in features:
            return

        feature_vec = features[commodity]

        # Update Beta priors (reward in [0, 1] after normalization)
        normalized_reward = max(0, min(1, (reward + 0.1) / 0.2))  # Map [-10%, +10%] to [0, 1]

        for i in range(self.feature_dim):
            self.alpha[commodity][i] += feature_vec[i] * normalized_reward
            self.beta[commodity][i] += feature_vec[i] * (1 - normalized_reward)

        # Update capital tracking
        self.capital *= (1 + reward)
        self.peak_capital = max(self.peak_capital, self.capital)

    def generate_weekly_report(self, week, selected, features, reward, override_info):
        """Generate structured weekly report."""
        report = {
            "week": week,
            "timestamp": datetime.now().isoformat(),
            "allocation": selected,
            "override": override_info[0],
            "override_reason": override_info[1],
            "weekly_return": reward,
            "capital": self.capital,
            "drawdown": (self.capital - self.peak_capital) / self.peak_capital,
            "features": {c: features[c].tolist() for c in features} if features else {},
            "disabled_commodities": list(self.disabled_commodities)
        }
        self.decision_log.append(report)
        return report

print("✓ CommodityAllocationSystem ready")

## Step 2: Load Commodity Data

Simulate commodity price data (in production, use yfinance or alternative data source).

In [ ]:
def generate_synthetic_commodity_data(commodities, weeks=52*3, seed=42):
    """Generate synthetic commodity price data.

    In production, replace with:
    import yfinance as yf
    data = yf.download(['GC=F', 'CL=F', 'NG=F', 'HG=F'], period='3y', interval='1wk')
    """
    np.random.seed(seed)
    dates = pd.date_range(end=datetime.now(), periods=weeks, freq='W')

    prices = {}
    for commodity in commodities:
        # Each commodity has different drift and volatility
        drift = np.random.uniform(-0.001, 0.003)
        vol = np.random.uniform(0.02, 0.04)

        returns = np.random.normal(drift, vol, weeks)
        prices[commodity] = 100 * np.exp(np.cumsum(returns))

    return pd.DataFrame(prices, index=dates)


# Generate data
commodities = ["GOLD", "OIL", "NATGAS", "COPPER"]
price_data = generate_synthetic_commodity_data(commodities)

print(f"✓ Loaded {len(price_data)} weeks of data for {len(commodities)} commodities")
print(f"\nData range: {price_data.index[0].date()} to {price_data.index[-1].date()}")
print(f"\nFirst 5 weeks:")
print(price_data.head())

## Step 3: Run Full 52-Week Simulation

In [ ]:
# Initialize system
system = CommodityAllocationSystem(commodities, initial_capital=100000)

# Historical simulation (52 weeks = 1 year)
simulation_weeks = 52
start_week = 52  # Start after enough history for features

weekly_allocations = []
weekly_returns = []
weekly_capital = []

print("Running 52-week commodity allocation simulation...\n")
print("=" * 70)

for week in range(start_week, start_week + simulation_weeks):
    # Extract features
    features = system.get_market_features(price_data, week)

    # Select commodity
    selected, scores = system.select_commodity(features)

    # Apply guardrails
    final_allocation, override, reason = system.apply_guardrails(selected, features, week)

    # Calculate weekly return
    if final_allocation == "CASH":
        weekly_return = 0.0
    else:
        # Return = (price[week+1] - price[week]) / price[week]
        if week + 1 < len(price_data):
            weekly_return = (price_data[final_allocation].iloc[week + 1] /
                           price_data[final_allocation].iloc[week] - 1)
        else:
            weekly_return = 0.0

    # Update system
    system.update(final_allocation, features, weekly_return)

    # Generate report
    report = system.generate_weekly_report(
        week - start_week + 1,
        final_allocation,
        features,
        weekly_return,
        (override, reason)
    )

    # Track for visualization
    weekly_allocations.append(final_allocation)
    weekly_returns.append(weekly_return)
    weekly_capital.append(system.capital)

    # Print summary every 10 weeks
    if (week - start_week) % 10 == 0:
        print(f"Week {week-start_week+1:2d}: {final_allocation:8s} | "
              f"Return: {weekly_return:+6.2%} | "
              f"Capital: ${system.capital:,.0f} | "
              f"Override: {override}")

print("=" * 70)
print(f"\n✓ Simulation complete")
print(f"Final capital: ${system.capital:,.2f} ({(system.capital/100000 - 1)*100:+.1f}%)")
print(f"Total alerts: {len(system.alerts)}")

## Step 4: Generate Weekly Report Dashboard

In [ ]:
print("\n" + "=" * 70)
print("WEEKLY ALLOCATION REPORT")
print("=" * 70)

# Latest report
latest = system.decision_log[-1]

print(f"\n📅 Week: {latest['week']}")
print(f"📊 Current Allocation: {latest['allocation']}")
print(f"💰 Capital: ${latest['capital']:,.2f}")
print(f"📈 This Week Return: {latest['weekly_return']:+.2%}")
print(f"📉 Drawdown from Peak: {latest['drawdown']:.2%}")

if latest['override']:
    print(f"⚠️  Override: {latest['override_reason']}")

# Allocation distribution
print(f"\n🎯 ALLOCATION HISTORY (Last 52 weeks)")
from collections import Counter
allocation_counts = Counter(weekly_allocations)
for commodity, count in sorted(allocation_counts.items(), key=lambda x: -x[1]):
    pct = 100 * count / len(weekly_allocations)
    bar = "█" * int(pct / 2)
    print(f"{commodity:8s} {bar:25s} {pct:5.1f}% ({count} weeks)")

# Performance metrics
total_return = (system.capital / 100000 - 1)
avg_weekly_return = np.mean(weekly_returns)
sharpe_ratio = np.mean(weekly_returns) / (np.std(weekly_returns) + 1e-8) * np.sqrt(52)

print(f"\n📊 PERFORMANCE METRICS")
print(f"Total Return: {total_return:+.2%}")
print(f"Avg Weekly Return: {avg_weekly_return:+.3%}")
print(f"Sharpe Ratio (annualized): {sharpe_ratio:.2f}")
print(f"Best Week: {max(weekly_returns):+.2%}")
print(f"Worst Week: {min(weekly_returns):+.2%}")

# Alerts
if system.alerts:
    print(f"\n🚨 ALERTS")
    for alert in system.alerts:
        print(f"   {alert}")

print("\n" + "=" * 70)

## Step 5: Visualize Full History

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

weeks = range(1, len(weekly_capital) + 1)

# Capital curve
axes[0].plot(weeks, weekly_capital, linewidth=2, color='#2E86AB')
axes[0].axhline(100000, color='gray', linestyle='--', alpha=0.5, label='Initial capital')
axes[0].set_title('Portfolio Capital Over Time', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Week')
axes[0].set_ylabel('Capital ($)')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].fill_between(weeks, 100000, weekly_capital, alpha=0.2)

# Weekly returns
colors = ['green' if r > 0 else 'red' for r in weekly_returns]
axes[1].bar(weeks, weekly_returns, color=colors, alpha=0.6)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Weekly Returns', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Week')
axes[1].set_ylabel('Return (%)')
axes[1].grid(axis='y', alpha=0.3)

# Allocation over time
commodity_to_num = {c: i for i, c in enumerate(commodities + ["CASH"])}
allocation_nums = [commodity_to_num[a] for a in weekly_allocations]
axes[2].scatter(weeks, allocation_nums, alpha=0.6, s=50, c=allocation_nums, cmap='tab10')
axes[2].set_title('Commodity Allocation Over Time', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Week')
axes[2].set_ylabel('Allocated Commodity')
axes[2].set_yticks(range(len(commodities) + 1))
axes[2].set_yticklabels(commodities + ["CASH"])
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Visualization complete")

## Step 6: Export Logs for Analysis

In [ ]:
# Export decision logs to JSON Lines format
log_file = '/tmp/commodity_allocation_log.jsonl'

with open(log_file, 'w') as f:
    for entry in system.decision_log:
        f.write(json.dumps(entry) + '\n')

print(f"✓ Exported {len(system.decision_log)} decision logs to {log_file}")

# Create summary CSV for easy analysis
summary_df = pd.DataFrame([
    {
        'week': log['week'],
        'allocation': log['allocation'],
        'return': log['weekly_return'],
        'capital': log['capital'],
        'override': log['override']
    }
    for log in system.decision_log
])

csv_file = '/tmp/commodity_allocation_summary.csv'
summary_df.to_csv(csv_file, index=False)
print(f"✓ Exported summary to {csv_file}")

print("\nSample log entry (first week):")
print(json.dumps(system.decision_log[0], indent=2))

## Step 7: Deployment Instructions

How to run this system in production with automated weekly execution.

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════════╗
║          PRODUCTION DEPLOYMENT INSTRUCTIONS                        ║
╚════════════════════════════════════════════════════════════════════╝

## Option 1: Cron Job (Simple)

1. Save this notebook as `commodity_allocator.py`
2. Add cron job to run every Friday at 4 PM ET:

   0 16 * * 5 cd /path/to/project && python commodity_allocator.py >> logs/allocation.log 2>&1

3. Monitor logs in `logs/allocation.log`

## Option 2: Airflow DAG (Recommended for Production)

```python
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta

def run_commodity_allocation():
    # Import and run allocation system
    from commodity_allocator import CommodityAllocationSystem
    system = CommodityAllocationSystem(...)
    # ... run weekly allocation

dag = DAG(
    'commodity_allocation',
    schedule_interval='0 16 * * 5',  # Friday 4 PM
    start_date=datetime(2026, 1, 1),
    catchup=False
)

allocate_task = PythonOperator(
    task_id='run_allocation',
    python_callable=run_commodity_allocation,
    dag=dag
)
```

## Option 3: Cloud Function (AWS Lambda / GCP Cloud Functions)

1. Package code + dependencies
2. Set up CloudWatch/Cloud Scheduler trigger for Fridays 4 PM ET
3. Store logs in S3/GCS
4. Send alerts to SNS/Pub/Sub

## Required Environment Variables

export ALLOCATION_CAPITAL=100000
export ALLOCATION_MAX_POSITION=0.40
export ALLOCATION_STOP_LOSS=-0.15
export ALLOCATION_LOG_PATH=/var/log/commodity_allocation/
export ALERT_EMAIL=your-email@example.com

## Pre-Deployment Checklist

✓ Backtest on minimum 2 years historical data
✓ Paper trade for 1 month (no real capital)
✓ Set up monitoring dashboard (Grafana/Datadog)
✓ Configure alert channels (email, Slack, PagerDuty)
✓ Test guardrail triggers (stop-loss, position limits)
✓ Prepare rollback plan (fallback to equal weights)
✓ Document decision rationale and risk disclosures
✓ Get approval from risk management team

## Monitoring URLs

- Logs: /var/log/commodity_allocation/
- Dashboard: http://localhost:3000/commodities (Grafana)
- Alerts: Check email or Slack #trading-alerts

## Emergency Contacts

- System Owner: [Your Name] - [Email]
- Risk Manager: [Name] - [Email]
- On-Call: [PagerDuty/Slack]

""")

## Modify This: Production Enhancements

Try these production improvements:
1. **Real data integration** — Replace synthetic data with yfinance
2. **Email alerts** — Send email when stop-loss triggers
3. **Slack reporting** — Post weekly summary to Slack channel
4. **Database storage** — Store logs in PostgreSQL instead of JSON
5. **Dashboard** — Build Streamlit/Gradio web dashboard for monitoring

In [ ]:
# Your production enhancements here

# Example: Add email alerting
def send_alert_email(subject, message):
    """Send email alert when critical event occurs."""
    # import smtplib
    # ... implementation
    pass

## Summary: Course Completion

**Congratulations!** You've completed the Multi-Armed Bandits & A/B Testing course.

**What you've learned:**

1. **Foundations** (Module 0)
   - Explore-exploit tradeoff and regret minimization
   - Why A/B testing leaves money on the table

2. **Core Algorithms** (Modules 1-3)
   - Epsilon-greedy, UCB, Thompson Sampling
   - Bayesian bandits with Beta priors
   - Contextual bandits (LinUCB, neural bandits)

3. **Applications** (Modules 4-5)
   - Content optimization and growth hacking
   - Commodity trading and portfolio allocation

4. **Advanced Topics** (Module 6)
   - Non-stationary bandits and drift detection
   - Combinatorial bandits and constraints

5. **Production Systems** (Module 7)
   - System architecture and separation of concerns
   - Logging, monitoring, and alerting
   - Offline evaluation with IPS/DR
   - A/B to bandit migration strategies
   - **End-to-end commodity allocation system (deployed!)**

**Your Production-Ready Toolkit:**
- Complete bandit engine with logging and monitoring
- Commodity allocation system ready for weekly deployment
- Guardrails for risk management
- Offline evaluation framework
- Migration playbook from A/B testing

**Next Steps:**
1. Deploy the commodity allocator with paper trading (no real capital)
2. Monitor for 1 month, validate behavior
3. Read advanced papers in `resources/additional_readings.md`
4. Build your own portfolio project applying bandits to your domain
5. Contribute to open-source bandit libraries (Vowpal Wabbit, Ray RLlib)

**Remember:** 
- Always backtest thoroughly before deploying with real capital
- Implement proper risk management (stop-loss, position limits)
- Monitor continuously, especially in the first month
- This course is educational — consult professionals before trading

**Thank you for completing this course!**

Share your projects, questions, and success stories in the course forum.
""")

In [ ]:
key_takeaways(["Real data integration", "Email alerts", "Slack reporting", "Database storage", "Dashboard"])